# Aggregate by river basin

The fleet's per-tile partial sums → the river-basin cube (HydroBASINS level 5: runoff-onset statistics per basin ×
100 m elevation × CHILI insolation class × water year), the per-basin ERA5-Land anomaly zonal means merged into it,
and the per-basin metrics table the two basin notebooks plot. The fleet keys basins at HydroBASINS level 6;
Pfafstetter codes nest by digit prefix, so the level-5 cube is the exact sum of the level-6 rows. Run it after the
three GitHub Actions workflows and before the other notebooks here.

| | |
| --- | --- |
| Reads | `partials/<version>/tile_*.parquet`, downloaded from Azure (`snowmelt_runoff_onset_analysis/partials/<version>/`) when the SAS token is available, otherwise the local cache as is; the BasinATLAS v1.0 level-5 polygons from the gdb cached in `data/geometries/` (2.7 GB, downloaded once); `data/geometries/hydrobasins_level6_population.csv` (tracked; `pixi run population` rebuilds it from Earth Engine); the ERA5-Land anomaly group on Azure |
| Writes | `data/aggregation/<version>/all_river_basins_<filter>.nc` (level 5, one cube per pixel filter), `era5_anomaly_river_basins.nc`, optionally the level-6 cube and its zonal means; `results/<version>/river_basin_metrics.csv` (tracked) |
| Needs | the Azure SAS token for the partials download and the ERA5-Land step; without it (the CI smoke test on two fixture tiles) the ERA5 step is skipped |

What a partials row is, why sums are enough, and what changed against the 2025 workflow: `pipeline/README.md`
and `docs/aggregation_lineage.md`.

In [ ]:
import time

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

from gsro_analysis import aggregate, era5, paths, results, settings

In [ ]:
config = settings.load_config()          # the dataset version lives in settings.CONFIG_FILE
VERSION = config.version
WATER_YEARS = [int(y) for y in config.water_years]
UNIT = 'river_basins'
FILTER_TAGS = list(aggregate.FILTERS)    # the pixel filters the fleet emitted: 'full_dataset' and 'fcf_lte_50' (the analyses' rule)
BUILD_LEVEL6_CUBE = False                # also write the HydroBASINS level-6 cube (~100 MB) and its zonal means
aggregation_dir = paths.aggregation_dir(UNIT, VERSION)   # analyses/river_basins/data/aggregation/<version>/
results_dir = paths.resultsdir(UNIT, VERSION)            # analyses/river_basins/results/<version>/

# the Azure SAS token is needed for the partials download and the ERA5-Land step; without it (the CI smoke test on the
# fixture tiles) the notebook keeps going: using the local partials cache as is, skipping the ERA5-Land section
try:
    config.sas_token
    HAVE_AZURE = True
except ValueError as e:
    HAVE_AZURE = False
    print(f'no Azure SAS token: using the local partials cache as is, skipping the ERA5-Land section ({e})')
print(f'{VERSION} | water years {WATER_YEARS[0]}-{WATER_YEARS[-1]} | filters {FILTER_TAGS} | Azure: {HAVE_AZURE}')
print(f'cubes -> {aggregation_dir}')

## 1. The fleet's partial sums, one parquet per tile

Every row is one tile's contribution to one cell of the cube: the pixels of one (filter, unit type, unit id,
elevation / aspect / latitude bin, CHILI class) with their count and the sums the statistics need
(Σ median, Σ median², per water year Σ onset, Σ onset², Σ anomaly, Σ anomaly², the CHILI and forest-cover
correlation sums). A unit that spans several tiles is several rows; adding them is the reduce.

In [ ]:
partials_dir = paths.partials_cache(VERSION)                        # partials/<version>/ (gitignored)
if HAVE_AZURE:
    partial_files = aggregate.sync_partials(config, partials_dir)   # downloads the tiles missing from the cache, drops stale ones
else:
    partial_files = sorted(partials_dir.glob('tile_*.parquet'))
print(f'{len(partial_files)} tiles in {partials_dir}')

In [ ]:
tile_partials_df = pd.read_parquet(partial_files[0])
print(f'{partial_files[0].name}: {len(tile_partials_df)} rows x {len(tile_partials_df.columns)} columns; '
      'one row = one tile\'s pixels in one (filter, unit type, unit id, bins, CHILI class) cell')
tile_partials_df

In [ ]:
# Sum the partials over tiles, keeping only this unit type. The reduce is a sum over identical keys, so
# summing BATCH tiles at a time gives the same result as concatenating everything first, at a fraction of the
# memory (the full campaign is ~14 M rows; a mountain-range tile alone is ~10 k rows). Each tile is cut down to
# this unit before it joins the batch. min_count=1 keeps a column NaN when no tile reported it.
KEY_COLS = ['filter_tag', 'unit_type', 'unit_id', 'elevation', 'aspect', 'latitude', 'chili_class']
BATCH = 50
t0 = time.time()
summed_partials_df, n_rows = None, 0
for i in range(0, len(partial_files), BATCH):
    tiles = []
    for f in partial_files[i:i + BATCH]:
        tile_df = pd.read_parquet(f)
        if 'unit_type' not in tile_df.columns:    # a verified-empty tile (no pixel with a valid median passed the filters)
            continue
        tiles.append(tile_df[tile_df['unit_type'] == UNIT].drop(columns=['tile_row', 'tile_col'], errors='ignore'))
    batch_df = pd.concat(tiles, ignore_index=True)
    n_rows += len(batch_df)
    batch_sums_df = batch_df.groupby(KEY_COLS, sort=False, dropna=False).sum(min_count=1)
    del tiles, batch_df
    if summed_partials_df is None:
        summed_partials_df = batch_sums_df
    else:
        summed_partials_df = (pd.concat([summed_partials_df, batch_sums_df])
                              .groupby(level=KEY_COLS, sort=False, dropna=False).sum(min_count=1))
summed_partials_df = summed_partials_df.reset_index()
print(f'{n_rows:,} {UNIT} partial rows from {len(partial_files)} tiles summed into {len(summed_partials_df):,} '
      f'cube cells ({time.time() - t0:.0f}s)')
summed_partials_df

## 2. The basin polygons and their population

BasinATLAS v1.0 (Linke et al. 2019) ships every HydroBASINS level in one gdb; level 5 is the cube's unit, level 6 what
the fleet stored. The gdb spells `ORDER` with a trailing underscore, and a handful of basins are multipart records
sharing one `PFAF_ID`, dissolved here to one row each. Population per basin comes from the tracked level-6 table
(GPW v4.11 summed on Earth Engine by `pipeline/scripts/get_basin_population.py`); level 5 is the sum of its level-6
children, i.e. the first five digits of `PFAF_ID`.

In [ ]:
basin_atlas_gdb = settings.basin_atlas_gdb()        # downloaded once into data/geometries/ (2.7 GB, md5-checked)
basins_gdf = gpd.read_file(basin_atlas_gdb, layer=settings.basin_atlas_layer(5)).rename(columns={'ORDER_': 'ORDER'})
basins_gdf = basins_gdf[['PFAF_ID', 'HYBAS_ID', 'MAIN_BAS', 'SUB_AREA', 'geometry']]
if basins_gdf['PFAF_ID'].duplicated().any():
    basins_gdf = basins_gdf.dissolve(by='PFAF_ID', aggfunc={'HYBAS_ID': 'first', 'MAIN_BAS': 'first', 'SUB_AREA': 'sum'}, as_index=False)
print(f'{len(basins_gdf)} level-5 basins, {basins_gdf["SUB_AREA"].sum() / 1e6:.1f} million km2')
basins_gdf

In [ ]:
population_level6_df = pd.read_csv(paths.geometries(UNIT) / 'hydrobasins_level6_population.csv', comment='#')
population_level5_df = (population_level6_df.assign(PFAF_ID=population_level6_df['PFAF_ID'] // 10)
                        .groupby('PFAF_ID', as_index=False)['total_population'].sum())
print(f'{len(population_level6_df)} level-6 basins -> {len(population_level5_df)} level-5 basins, '
      f'{population_level5_df["total_population"].sum() / 1e9:.2f} billion people')
population_level5_df

## 3. ERA5-Land anomaly zonal means per basin (needs the Azure token)

For every basin, water year and hemisphere-aware month, the mean of the ERA5-Land monthly anomaly (each of the 8
variables minus its per-pixel median over all water years) over the 0.1° cells the polygon covers, each cell weighted
by its coverage fraction × its seasonal-snow fraction × whether the dataset has an onset value there that year (both
masks from the public pyramid). `era5.zonal_anomalies` does this for all basins in one streamed pass over the store.

In [ ]:
LEVELS = {'river_basins': 5} | ({'river_basins_l6': 6} if BUILD_LEVEL6_CUBE else {})
REBUILD_ERA5_ZONAL = False      # True recomputes even if the files exist (after an ERA5-Land rebuild)
era5_zonal = {}
for group, level in LEVELS.items():
    era5_zonal_path = aggregation_dir / f'era5_anomaly_{group}.nc'
    if HAVE_AZURE and (REBUILD_ERA5_ZONAL or not era5_zonal_path.exists()):
        if level == 5:
            polygons_gdf = basins_gdf
        else:
            polygons_gdf = gpd.read_file(basin_atlas_gdb, layer=settings.basin_atlas_layer(level))
            if polygons_gdf['PFAF_ID'].duplicated().any():
                polygons_gdf = polygons_gdf.dissolve(by='PFAF_ID', as_index=False)
        era5_anomaly_ds = era5.open_anomaly(config)     # the anomaly group of the version's ERA5-Land icechunk repository
        zonal_ds = era5.zonal_anomalies(config, polygons_gdf, 'PFAF_ID', anomaly_ds=era5_anomaly_ds)
        zonal_ds.attrs['unit_type'] = group
        zonal_ds.to_netcdf(era5_zonal_path, encoding={v: {'zlib': True, 'complevel': 4} for v in zonal_ds.data_vars})
        print(f'wrote {era5_zonal_path}')
    if era5_zonal_path.exists():
        era5_zonal[group] = xr.open_dataset(era5_zonal_path)
    else:
        print(f'no {era5_zonal_path.name}: the {group} cube gets no climate variables (rerun this section with the Azure token)')
era5_zonal.get('river_basins')

## 4. The cube, one file per pixel filter

`aggregate.reduce_partials` truncates the stored level-6 ids to level 5, sums the rows per basin, and turns the sums
into bin means (Σx / n), standard deviations (√(Σx² / n − mean²)), pixel counts and the CHILI / forest-cover
correlations on the dense `river_basin × elevation × chili_class × water_year` grid; the ERA5-Land zonal means are
merged on `(river_basin, water_year, month)`.

In [ ]:
for group in LEVELS:
    for filter_tag in FILTER_TAGS:
        t0 = time.time()
        try:
            cube_ds = aggregate.reduce_partials(summed_partials_df, group, filter_tag, WATER_YEARS)
        except ValueError as e:               # no rows for this filter (a partially processed version)
            print(f'{group}/{filter_tag}: {e}')
            continue
        if group in era5_zonal:
            zonal_ds = era5_zonal[group].rename({'PFAF_ID': 'river_basin'})
            zonal_ds = zonal_ds.sel(river_basin=zonal_ds['river_basin'].isin(cube_ds['river_basin'].values))
            cube_ds = xr.merge([cube_ds, zonal_ds], combine_attrs='drop_conflicts', join='left')
        cube_ds.attrs.update({'dataset_version': VERSION, 'n_tiles': len(partial_files),
                              'produced_by': 'analyses/river_basins/0_aggregate_by_river_basin.ipynb'})
        cube_path = aggregation_dir / f'all_{group}_{filter_tag}.nc'
        encoding = {v: {'zlib': True, 'complevel': 4, **({'dtype': 'float32'} if cube_ds[v].dtype.kind == 'f' else {})}
                    for v in cube_ds.data_vars}
        cube_ds.to_netcdf(cube_path.with_suffix('.nc.tmp'), encoding=encoding)
        cube_path.with_suffix('.nc.tmp').replace(cube_path)
        print(f'wrote {cube_path.name}: {cube_path.stat().st_size / 1e6:.1f} MB, dims {dict(cube_ds.sizes)} ({time.time() - t0:.0f}s)')

In [ ]:
river_basins_ds = xr.open_dataset(aggregation_dir / 'all_river_basins_fcf_lte_50.nc')
river_basins_ds

## 5. The per-basin metrics table

One row per level-5 basin, written to `results/<version>/river_basin_metrics.csv` with provenance columns: basin
area (BasinATLAS `SUB_AREA`), population, mapped pixels and their share of the basin area, the pixel-weighted mean
median onset and MAD, and the pixel-weighted mean onset and anomaly per water year. The means are masked where less
than `MIN_AREA_PCT` of the basin is mapped, the yearly ones where less than `MIN_YEAR_AREA_PCT` has data that year —
the rule the basin maps have always used. `basin_onset.ipynb` and `snow_water.ipynb` join this table to the polygons.

In [ ]:
MIN_AREA_PCT = 5              # a basin's static means count only if more than 5 % of its area is mapped
MIN_YEAR_AREA_PCT = 1         # a basin-year's means only if more than 1 % of the area has data that year
PIXEL_AREA_KM2 = (80 / 1000) ** 2
BIN_DIMS = ['elevation', 'chili_class']

basin_ids = river_basins_ds['river_basin'].values
metrics_df = pd.DataFrame({'PFAF_ID': basin_ids}).set_index('PFAF_ID')
metrics_df['basin_area_km2'] = basins_gdf.set_index('PFAF_ID')['SUB_AREA'].reindex(basin_ids).values
metrics_df['population'] = population_level5_df.set_index('PFAF_ID')['total_population'].reindex(basin_ids).values
pixel_count_da = river_basins_ds['runoff_onset_median_n'].sum(BIN_DIMS)
metrics_df['pixel_count'] = pixel_count_da.values
metrics_df['area_pct'] = 100 * metrics_df['pixel_count'] * PIXEL_AREA_KM2 / metrics_df['basin_area_km2']
mapped = metrics_df['area_pct'] > MIN_AREA_PCT
metrics_df['runoff_onset_median'] = aggregate.weighted_mean(river_basins_ds, 'runoff_onset_median', BIN_DIMS).values
metrics_df['runoff_onset_mad'] = aggregate.weighted_mean(river_basins_ds, 'runoff_onset_mad', BIN_DIMS).values
metrics_df.loc[~mapped, ['runoff_onset_median', 'runoff_onset_mad']] = np.nan
yearly_onset_da = aggregate.weighted_mean(river_basins_ds, 'runoff_onset', BIN_DIMS)
yearly_anomaly_da = aggregate.weighted_mean(river_basins_ds, 'runoff_onset_anomaly', BIN_DIMS)
yearly_count_da = river_basins_ds['runoff_onset_n'].sum(BIN_DIMS)
for year in WATER_YEARS:
    year_pct = 100 * yearly_count_da.sel(water_year=year).values * PIXEL_AREA_KM2 / metrics_df['basin_area_km2']
    year_ok = (year_pct > MIN_YEAR_AREA_PCT).values
    metrics_df[f'runoff_onset_WY{year}'] = np.where(year_ok, yearly_onset_da.sel(water_year=year).values, np.nan)
    metrics_df[f'runoff_onset_anomaly_WY{year}'] = np.where(year_ok, yearly_anomaly_da.sel(water_year=year).values, np.nan)
    metrics_df[f'pixel_count_WY{year}'] = yearly_count_da.sel(water_year=year).values
print(f'{len(metrics_df)} basins, {int(mapped.sum())} with more than {MIN_AREA_PCT} % of their area mapped')
metrics_df

In [ ]:
metrics_path = results_dir / 'river_basin_metrics.csv'
metrics_df = metrics_df.reset_index().round(3).assign(**results.provenance(config))   # _version, _git_sha, _analysis_git_sha, _written_at
metrics_df.to_csv(metrics_path, index=False)
print(f'{len(metrics_df)} basins x {len(metrics_df.columns)} columns -> {metrics_path}')
metrics_df.head()

In [ ]:
# a first look: the basin-mean median onset against the basin's pixel-weighted mean MAD, mapped basins only
f, ax = plt.subplots(figsize=(6, 5))
metrics_df.plot.scatter(ax=ax, x='runoff_onset_median', y='runoff_onset_mad', s=6, alpha=0.6)
ax.set_xlabel('basin-mean median runoff onset [day of water year]')
ax.set_ylabel('basin-mean MAD of runoff onset [days]')